In [99]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cobra
from cobra.core.configuration import Configuration
from cobra.io import read_sbml_model, write_sbml_model
from cobra.flux_analysis import flux_variability_analysis
from tqdm import tqdm

In [100]:
M_xanthus = read_sbml_model("../M_xanthus_model_V3.xml")
M_xanthus

Name,myxo_model
Memory address,72929149a5d0
Number of metabolites,1223
Number of reactions,1337
Number of genes,1200
Number of groups,0
Objective expression,1.0*OF_BIOMASS - 1.0*OF_BIOMASS_reverse_80d2e
Compartments,"c, e"


In [101]:
M_xanthus.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
Fe3_e,EX_Fe3_e,1.061,0,0.00%
ac_e,EX_ac_e,119.4,2,1.82%
alaala_e,EX_alaala_e,507.1,6,23.20%
ca2_e,EX_ca2_e,0.3535,0,0.00%
cl_e,EX_cl_e,0.3535,0,0.00%
cobalt2_e,EX_cobalt2_e,0.3535,0,0.00%
cu2_e,EX_cu2_e,0.3535,0,0.00%
glu_L_e,EX_glu_L_e,41.29,5,1.57%
his_L_e,EX_his_L_e,16.9,6,0.77%
ile_L_e,EX_ile_L_e,51.56,6,2.36%


In [102]:
iMAT_res = pd.read_csv("/home/mickael/github/M_xanthus-E_coli-Predation/results/quantiles/iMAT/log2FoldChange/epsilon_1.0_quantiles_40_70_name.csv", sep=";", index_col="Unnamed: 0")
iMAT_res

,reaction_id,flux_value,classification,y_f,y_r
0,"2-amino-4-hydroxy-6-hydroxymethyl-7,8-dihydrop...",0.0,moderate,NaN,NaN
1,gamma-L-glutamyl-L-cysteine:glycine ligase (AD...,1.0,high,1.0,0.0
2,R07600 [c],0.0,low,1.0,0.0
3,IMP:diphosphate phospho-D-ribosyltransferase [c],0.0,moderate,NaN,NaN
4,"2,3-dihydro-2,3-dihydroxybenzoate:NAD+ oxidore...",0.0,low,1.0,0.0
...,...,...,...,...,...
1362,Exchange for D-Lactate [e],255.0,NaN,NaN,NaN
1363,Exchange for H2S2O3 [e],0.0,NaN,NaN,NaN
1364,Demand for glycogen(n-1) [c],0.0,NaN,NaN,NaN
1365,Demand for Biomass [c],0.0,NaN,NaN,NaN


In [103]:
iMAT_res_filter = iMAT_res[iMAT_res.flux_value != 0]
iMAT_res_filter

,reaction_id,flux_value,classification,y_f,y_r
1,gamma-L-glutamyl-L-cysteine:glycine ligase (AD...,1.000000,high,1.0,0.0
12,L-glutamate:L-cysteine gamma-ligase (ADP-formi...,1.000000,high,1.0,0.0
14,ATP:dCDP phosphotransferase [c],1.000000,high,1.0,0.0
26,oxalosuccinate carboxy-lyase (2-oxoglutarate-f...,501.142860,moderate,NaN,NaN
31,Sulfate adenyltransferase [c],1.000000,high,1.0,0.0
...,...,...,...,...,...
1341,Exchange for Enterobactin [e],1000.000000,NaN,NaN,NaN
1353,Exchange for NH3 [e],5.785714,NaN,NaN,NaN
1357,Exchange for Acetoacetate [e],-754.214290,NaN,NaN,NaN
1360,Exchange for O2 [e],-870.500000,NaN,NaN,NaN


In [104]:
# Create dictionary to convert name to id
Dico_reaction = {}
for i in M_xanthus.reactions._dict:
    Dico_reaction[M_xanthus.reactions.get_by_id(i).name] = i
print(Dico_reaction)

# create a list of reaction id that should have flux
active_list = []
for i in iMAT_res_filter["reaction_id"]:
    active_list.append(Dico_reaction[i])
print(active_list)

{'2-amino-4-hydroxy-6-hydroxymethyl-7,8-dihydropteridine-diphosphate:4-aminobenzoate 2-amino-4-hydroxydihydropteridine-6-methenyltransferase [c]': 'rxn02201_c', 'gamma-L-glutamyl-L-cysteine:glycine ligase (ADP-forming) [c]': 'rxn00351_c', 'R07600 [c]': 'rxn07431_c', 'IMP:diphosphate phospho-D-ribosyltransferase [c]': 'rxn00836_c', '2,3-dihydro-2,3-dihydroxybenzoate:NAD+ oxidoreductase [c]': 'rxn01094_c', 'acetyl-CoA:L-serine O-acetyltransferase [c]': 'rxn00423_c', 'palmitoyl-lipoteichoic acid synthesis (n=24), linked, glucose substituted [c]': 'rxn10298_c', 'ATP:CMP phosphotransferase [c]': 'rxn00364_c', 'Transport of dicarboxylates, extracellular [c]': 'rxn05561_c', 'UDP-N-acetyl-D-glucosamine:undecaprenyl-diphospho-N-acetylmuramoyl-L-alanyl-gamma-D-glutamyl-meso-2,6-diaminopimeloyl-D-alanyl-D-alanine 4-beta-N-acetylglucosaminlytransferase [c]': 'rxn03408_c', 'Isochorismate pyruvate-hydrolase [c]': 'rxn02177_c', 'FACOAL160(ISO) [c]': 'rxn05250_c', 'L-glutamate:L-cysteine gamma-ligase 

Force to activate specific reaction

In [105]:
FBA = M_xanthus.optimize()
FBA.fluxes["rxn09240_c"]

np.float64(0.0)

In [137]:
# TODO add constraint that force the fluxe to pass there / Not work
M_xanthus.reactions.get_by_id("rxn09240_c")
have_flux = M_xanthus.problem.Constraint(
    M_xanthus.reactions.rxn09240_c.flux - 0,
    lb=0,
    ub=0)
M_xanthus.add_cons_vars(have_flux)

In [130]:
FBA = M_xanthus.optimize()
FBA.fluxes["rxn09240_c"]

np.float64(0.0)

Shut down all others reactions

In [110]:
for i in M_xanthus.reactions:
    if i.id not in active_list:
        i.bounds = [0,0]

M_xanthus.slim_optimize() # Didn't growth

-2.1126720371209696e-12